###### The University of Melbourne, School of Computing and Information Systems
# COMP30027 Machine Learning, 2025 sem 1

## Week 2 Tutorial

After working through this tutorial you should know

- how to define a real-world problem as a task for a machine learning algorithm
- how to open and read in a data set from a csv file
- how to split the data set into attributes (i.e., input to your ML algorithm) and labels (i.e., desired output of your ML algorithm)
- how to map string values to numeric values
- how to import and use a library in Python

## Machine learning basics

Considering the following problems:
1. Skin cancer screening test
2. Building a system that guesses what the weather (temperature, precipitation, etc.) will be like
tomorrow
3. Predicting products that a customer would be interested in buying, based on other purchases
that customer has previously made

### Q1
For each problem, what is the "concept" we might attempt to learn? (What is the task for the machine learning model?)

i) Skin Cancer Screening Test
    - Whether the patient has skin cancer or not (Binary Classification).

ii) Building a system that guesses what the weather will be like tomorrow
    - Numerical prediction of the weather temperature (Regression).

iii) Predicting products that a customer would be interested in buying, based on other purchases that customer has previously made
    - Is the customer interested or not interested in the product based on the attributes (Classification).
    - Seeing how similar customer attributes are to suggestion products (Clustering).
    - Based on past purchases whether the customer will be interested in new products (Association rule mining).

### Q2
For each problem-task, what are the instances and what attributes might be used to train the model?

i) Skin Cancer Screening Test
    - Instance: Patient
    - Attribute: Blood test, images from the skin, reports, observed syndromes, ...

ii) Building a system that guesses what the weather will be like tomorrow
    - Instance: A day
    - Attribute: Temperature, precipitation, humidity, wind speed, ...

iii) Predicting products that a customer would be interested in buying, based on other purchases that customer has previously made
    - Instance: Customer
    - Attribute: Salary, past purchase history, nationality (different culture) etc ...

### Q3
For each problem-task, would you most likely adopt a supervised or unsupervised machine learning method?

i) Skin Cancer Screening Test
    - Supervised Learning: Classification (Yes/No)

ii) Building a system that guesses what the weather will be like tomorrow
    - Supervised Learning: Regression (Numerical analysis/prediction), Classification

iii) Predicting products that a customer would be interested in buying, based on other purchases that customer has previously made
    - Unsupervised Learning: Clustering, Association rule mining
    - Supervised Learning: Classification

### Q4
For each problem-task, how easy or difficult it would be to make a model that generalizes
to new cases? For example, could you predict the weather in any city in the world, or just in one
specific city?

i) Skin Cancer Screening Test
    - Difficult to generalise as real world training data often have biases. 
    - For example, we know skin cancer risk increases with age, so these variables will probably be correlated in your taining set.
        - Does this mean that skin cancer risk decreases with just age as a factor?

ii) Building a system that guesses what the weather will be like tomorrow
    - Generalising is difficult as depending on the geographical location the weather factors may change.

iii) Predicting products that a customer would be interested in buying, based on other purchases that customer has previously made
    - Diver range of personal preference and interest. One might be interested in sports product and others might be interested in gaming etc..
    - A customer model trained in one country might not generalise to other countries.
    - If it mostly learns everyday shopping patterns, it probably won't give good predictions for outlier situations like holiday purchasing. 

### Q5
What kinds of assumptions might a machine learning model make when tackling these problems?

Every model makes assumptions about the world and how the concepts we want to learn relates to the attributes of the data.
 - The concept is actually related to the attributes.

## Reading and Processing Data

*This code is deliberately written to be easy to understand, minimizing the use of libraries, syntactic sugar etc.*

In this section we are refreshing your knowledge about Python Series and DataFrames.

In [1]:
# let's read the data into a list of lines
data = open('weather_data.csv', 'r').readlines()

# we know that the first line is the header (variable names), the rest of the lines actually contain data
header = data[0]
instances = data[1:]

In [2]:
# let's look at the variables (columns) in this dataset
print(header)

outlook,temperature,humidity,windy,play



In [3]:
# let's look at the instances (rows)
print(instances)

['sunny,hot,high,FALSE,no\n', 'sunny,hot,high,TRUE,no\n', 'overc,hot,high,FALSE,yes\n', 'rainy,mild,high,FALSE,yes\n', 'rainy,cool,normal,FALSE,yes\n', 'rainy,cool,normal,TRUE,no\n', 'overc,cool,normal,TRUE,yes\n', 'sunny,mild,high,FALSE,no\n', 'sunny,cool,normal,FALSE,yes\n', 'rainy,mild,normal,FALSE,yes\n', 'sunny,mild,normal,TRUE,yes\n', 'overc,mild,high,TRUE,yes\n', 'overc,hot,normal,FALSE,yes\n', 'rainy,mild,high,TRUE,no\n']


### Q6
What does each instance represent? What is the task?

Let's use the list of instances, and create from it a list of attributes (x), and a list of labels (y)

In [4]:
# first, initialize the empty lists
features = []
labels = []

# iterate over our instances:
for instance in instances:
    
    instance = instance.strip() #remove all leading and trailing whitespace (i.e., the newline symbol '\n')
    instance = instance.split(",") # split each instance at each comma, into separate values
    
    inst_features = instance[:4]   # store the first 4 fields as the instance's features
   
    inst_label = instance[-1]    # store the label as the last field 
    
    # append this instance's to our global list of features / labels
    features.append(inst_features)
    labels.append(inst_label)

Let's look at what we got

In [5]:
print("all features: {}\n".format(features))
print("all labels  : {}\n".format(labels))

# print features and label of 1st instance
print("features of first instance: {}\nlabel of first instance: {} ".format(features[0], labels[0]))

all features: [['sunny', 'hot', 'high', 'FALSE'], ['sunny', 'hot', 'high', 'TRUE'], ['overc', 'hot', 'high', 'FALSE'], ['rainy', 'mild', 'high', 'FALSE'], ['rainy', 'cool', 'normal', 'FALSE'], ['rainy', 'cool', 'normal', 'TRUE'], ['overc', 'cool', 'normal', 'TRUE'], ['sunny', 'mild', 'high', 'FALSE'], ['sunny', 'cool', 'normal', 'FALSE'], ['rainy', 'mild', 'normal', 'FALSE'], ['sunny', 'mild', 'normal', 'TRUE'], ['overc', 'mild', 'high', 'TRUE'], ['overc', 'hot', 'normal', 'FALSE'], ['rainy', 'mild', 'high', 'TRUE']]

all labels  : ['no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'yes', 'no']

features of first instance: ['sunny', 'hot', 'high', 'FALSE']
label of first instance: no 


## Converting attributes from strings to numbers

Now, computers are much better at working with numbers than with strings. Let's write a function that maps each type of value to a unique number. We can do this by

1. creating a set of all occuring values (a set by definition contains each value exactly once)
2. map each value to its position in this list

For example
- our observed values are v=[a,b,c,a,a,b,d]
- turning this into a set: set(v)=[a,b,c,d]
- and turning each value into a number based on its set position: a=0, b=1, c=2, d=4

In [6]:
def string_feature_to_numeric_feature(str_values):
    str_value_set = list(set(str_values)) # create a set of all values in value_list
    numeric_values = [] # initialize our new value list
    
    for str_value in str_values:
        
        num_value = str_value_set.index(str_value)    # indexing element's position of str_value

        numeric_values.append(num_value) # append the numeric value to the new value list
    
    return numeric_values # return the new numeric values as an output of the function

Let's see if it works

In [7]:
numeric_labels = string_feature_to_numeric_feature(labels)

print("string labels : {}".format(labels))
print("numeric labels: {}".format(numeric_labels))

string labels : ['no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'yes', 'no']
numeric labels: [0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0]



Similary, we can iterate over all features (columns in our data matrix), and transform all our features and labels to numeric values

In [8]:
# print features
print("Features LOOK: ", features)

# print the original features
print("\nThis is our original feature matrix:")
for i in features:
    print('\t'.join(i))

# initialize our new structure to hold the numeric features
# numeric_features = [[] for i in features]
numeric_features = []
for i in features:
    numeric_features.append([])

# print("\n Numeric Features LOOK: ", numeric_features)

# iterate over each feature (i.e., over the columns of our data set)
for feature_idx in range(len(features[0])):
    
    str_feat_values = []    
    for values in features:
        str_feat_values.append(values[feature_idx])

    print("\n Str_feat_values LOOK: ", str_feat_values)
    
    num_feat_values = string_feature_to_numeric_feature(str_feat_values)

    print("\n num_feat_values LOOK: ", num_feat_values)

    for idx, instance in enumerate(features):
        numeric_features[idx].append(num_feat_values[idx])
        print("--------------------------------------------------")
        print("\n LOOK idx: ", idx)
        print("\n LOOK instance: ", instance)
        print("\n Numeric Features LOOK: ", numeric_features)
    
# print the new, numeric veatures
print("\n\nThis is our new, numeric feature matrix:")
for i in numeric_features:
    print('{}\t{}\t{}\t{}'.format(i[0], i[1], i[2], i[3]))

Features LOOK:  [['sunny', 'hot', 'high', 'FALSE'], ['sunny', 'hot', 'high', 'TRUE'], ['overc', 'hot', 'high', 'FALSE'], ['rainy', 'mild', 'high', 'FALSE'], ['rainy', 'cool', 'normal', 'FALSE'], ['rainy', 'cool', 'normal', 'TRUE'], ['overc', 'cool', 'normal', 'TRUE'], ['sunny', 'mild', 'high', 'FALSE'], ['sunny', 'cool', 'normal', 'FALSE'], ['rainy', 'mild', 'normal', 'FALSE'], ['sunny', 'mild', 'normal', 'TRUE'], ['overc', 'mild', 'high', 'TRUE'], ['overc', 'hot', 'normal', 'FALSE'], ['rainy', 'mild', 'high', 'TRUE']]

This is our original feature matrix:
sunny	hot	high	FALSE
sunny	hot	high	TRUE
overc	hot	high	FALSE
rainy	mild	high	FALSE
rainy	cool	normal	FALSE
rainy	cool	normal	TRUE
overc	cool	normal	TRUE
sunny	mild	high	FALSE
sunny	cool	normal	FALSE
rainy	mild	normal	FALSE
sunny	mild	normal	TRUE
overc	mild	high	TRUE
overc	hot	normal	FALSE
rainy	mild	high	TRUE

 Str_feat_values LOOK:  ['sunny', 'sunny', 'overc', 'rainy', 'rainy', 'rainy', 'overc', 'sunny', 'sunny', 'rainy', 'sunny', 

## Opening and reading csv files with Python's Pandas library

There are various useful libraries which allow you to handle data sets much more efficiently (even though everything they do you could implement yourself fairly easily, similarly to the code you wrote above). One useful library is <b>Pandas</b>. Below is some Pandas example code.

In [9]:
import pandas as pd

data_p = pd.read_csv('weather_data.csv', sep=',')

print(data_p.head())

  outlook temperature humidity  windy play
0   sunny         hot     high  False   no
1   sunny         hot     high   True   no
2   overc         hot     high  False  yes
3   rainy        mild     high  False  yes
4   rainy        cool   normal  False  yes


In [10]:
label_p = data_p['play']
print(label_p)

0      no
1      no
2     yes
3     yes
4     yes
5      no
6     yes
7      no
8     yes
9     yes
10    yes
11    yes
12    yes
13     no
Name: play, dtype: object


In [11]:
features_p = data_p[['outlook', 'temperature', 'humidity', 'windy']]
print(features_p)

   outlook temperature humidity  windy
0    sunny         hot     high  False
1    sunny         hot     high   True
2    overc         hot     high  False
3    rainy        mild     high  False
4    rainy        cool   normal  False
5    rainy        cool   normal   True
6    overc        cool   normal   True
7    sunny        mild     high  False
8    sunny        cool   normal  False
9    rainy        mild   normal  False
10   sunny        mild   normal   True
11   overc        mild     high   True
12   overc         hot   normal  False
13   rainy        mild     high   True


The Pandas library includes many helpful functions for manipulating data. For example, here's how to turn string features into numeric features with minimal code:

1. Pandas 'apply' function which allows you to apply an operation to all items in the input dataframe
2. Pandas 'factorize' which automatically maps each categorical value to a unique integer it returns both the converted values, and the mapping it used. We are only interested in the converted values (hence the index [0])
3. Python's lambda functionality 'lambda i: expression' which executes 'expression' any number of input arguments (here: columns)

In [12]:
numeric_features_p = features_p.apply(lambda feature: pd.factorize(feature)[0])
print(numeric_features_p)

numeric_labels_p = pd.factorize(label_p)[0]
print(numeric_labels_p)

    outlook  temperature  humidity  windy
0         0            0         0      0
1         0            0         0      1
2         1            0         0      0
3         2            1         0      0
4         2            2         1      0
5         2            2         1      1
6         1            2         1      1
7         0            1         0      0
8         0            2         1      0
9         2            1         1      0
10        0            1         1      1
11        1            1         0      1
12        1            0         1      0
13        2            1         0      1
[0 0 1 1 1 0 1 0 1 1 1 1 1 0]
